# Exploring the Allmaps APIs from a IIIF manifest

[Allmaps](https://allmaps.org/) is an open source project that lets people georeference (align to real-world coordinates) scanned historical maps that are published through [IIIF](https://iiif.io/) (International Image Interoperability Framework). This notebook starts from nothing but a IIIF manifest URL and walks through the Allmaps APIs step by step, showing how to:

1. Ask Allmaps whether it already has a georeference annotation for a manifest.
2. Pull the Allmaps Map ID out of that annotation.
3. Build the URLs for the Allmaps Editor (to georeference a map), Allmaps Viewer (to view a georeferenced map), and Allmaps XYZ tile server (to use the map as a tile layer, e.g. in Leaflet or MapLibre).
4. Use the Allmaps REST API to look up map and manifest metadata directly.

Each code cell below is preceded by a short explanation, and every line inside the code is commented for readers who are new to Python and/or to spatial data APIs.


In [9]:
import requests
import json

# define manifest
# hard coded for this example

manifest = "https://purl.stanford.edu/tr202jn9182/iiif/manifest"

# print the georef annotation for that manifest as JSON data
# pass the manifest as a params value so requests URL-encodes it correctly

allmapsAPIRequest = requests.get("https://annotations.allmaps.org/", params={"url": manifest})

if allmapsAPIRequest.status_code >= 500:
    # A 500 here usually means Allmaps could not fetch/parse the manifest itself.
    # Check directly whether the manifest URL is reachable by a plain HTTP client.
    manifestCheck = requests.get(manifest)
    if "text/html" in manifestCheck.headers.get("Content-Type", ""):
        raise RuntimeError(
            "The manifest URL returned an HTML page instead of IIIF JSON "
            "(likely a bot/reCAPTCHA verification gate), so Allmaps could not "
            "fetch or parse it either. Use a manifest URL that is reachable "
            "without a browser session, e.g. the LMEC example later in this notebook."
        )

allmapsAPIRequest.raise_for_status()
georefAnnotation = allmapsAPIRequest.json()


print(georefAnnotation)


{'id': 'https://annotations.allmaps.org/manifests/cd78c03407a95d51', 'type': 'AnnotationPage', '@context': 'http://www.w3.org/ns/anno.jsonld', 'items': [{'id': 'https://annotations.allmaps.org/maps/a4c21a56d19cf41c', 'type': 'Annotation', '@context': ['http://iiif.io/api/extension/georef/1/context.json', 'http://iiif.io/api/presentation/3/context.json'], 'created': '2026-08-24T17:27:48.857Z', 'modified': '2026-08-24T17:27:48.857Z', 'motivation': 'georeferencing', 'target': {'type': 'SpecificResource', 'source': {'id': 'https://stacks.stanford.edu/image/iiif/tr202jn9182%2FAM_0228', 'type': 'ImageService2', 'height': 8638, 'width': 10261, 'partOf': [{'id': 'https://purl.stanford.edu/tr202jn9182/iiif/canvas/cocina-fileSet-tr202jn9182-c23a15fa-ab6e-4906-b112-bf410971384e', 'type': 'Canvas', 'label': {'none': ['Image 1']}, 'partOf': [{'id': 'https://purl.stanford.edu/tr202jn9182/iiif/manifest', 'type': 'Manifest', 'label': {'none': ['South Africa, from official & other authentic authorities

In [10]:
# detect Allmaps Map ID in georef anno

mapID = None

# handle both successful payload shapes and error payloads
items = georefAnnotation.get("item") or georefAnnotation.get("items")

if isinstance(items, dict):
    items = [items]

if items:
    raw_id = items[0].get("id", "")
    mapID = raw_id.rsplit("/", 1)[-1]  # keep only the map ID
else:
    raise ValueError(f"Could not find map ID. API response: {georefAnnotation}")

print(mapID)

a4c21a56d19cf41c


In [11]:
# construct georef annotation URL from map ID

annoBaseUrl = "https://annotations.allmaps.org/maps/"
georefAnnoUrl = annoBaseUrl+mapID

print(georefAnnoUrl)

https://annotations.allmaps.org/maps/a4c21a56d19cf41c


In [12]:
# now we have a georef anno endpoint + map ID just from the IIIF manifest

print("IIIF Manifest URL: "+manifest+"\n\r")
print("Map ID: "+mapID+"\n\r")
print("Georeference Annotation URL: "+georefAnnoUrl+"\n\r")


IIIF Manifest URL: https://purl.stanford.edu/tr202jn9182/iiif/manifest

Map ID: a4c21a56d19cf41c

Georeference Annotation URL: https://annotations.allmaps.org/maps/a4c21a56d19cf41c



In [13]:
# construct Editor endpoint with URL parameter

editorBaseUrl = "https://editor.allmaps.org/#/collection?url="
editorEndpoint = editorBaseUrl+manifest

print(editorEndpoint)

https://editor.allmaps.org/#/collection?url=https://purl.stanford.edu/tr202jn9182/iiif/manifest


In [14]:
# optionally add a callback URL to the header of Editor
# only works with certain institutions currently
# which is why example uses LMEC map

lmecObject = "https://collections.leventhalmap.org/search/commonwealth:0z709604j"
lmecManifest = lmecObject+"/manifest"
allmapsCallbackURL = "https://editor.allmaps.org/#/collection?url=" + lmecManifest + "&callback="+ lmecObject;

print(allmapsCallbackURL)


https://editor.allmaps.org/#/collection?url=https://collections.leventhalmap.org/search/commonwealth:0z709604j/manifest&callback=https://collections.leventhalmap.org/search/commonwealth:0z709604j


In [15]:
# construct Viewer endpoint with URL parameter

georefAnnotationUrlParam = "https://annotations.allmaps.org/?url="+manifest
viewerBaseUrl = "https://viewer.allmaps.org/?url="
viewerEndpointUrlParam = viewerBaseUrl+georefAnnotationUrlParam

print(viewerEndpointUrlParam)

https://viewer.allmaps.org/?url=https://annotations.allmaps.org/?url=https://purl.stanford.edu/tr202jn9182/iiif/manifest


In [16]:
# construct Viewer endpoint with pure Georeference Annotation

viewerEndpointPureAnno = viewerBaseUrl+georefAnnoUrl

print(viewerEndpointPureAnno)

https://viewer.allmaps.org/?url=https://annotations.allmaps.org/maps/a4c21a56d19cf41c


In [17]:
# construct XYZ tile endpoint from URL param

xyzBaseUrl = "https://allmaps.xyz/{z}/{x}/{y}.png?url="
xyzEndpointUrlParam = xyzBaseUrl+georefAnnoUrl

print(xyzEndpointUrlParam)

https://allmaps.xyz/{z}/{x}/{y}.png?url=https://annotations.allmaps.org/maps/a4c21a56d19cf41c


In [18]:
# construct XYZ tile endpoint from map ID

print(f"https://allmaps.xyz/maps/{mapID}/{{z}}/{{x}}/{{y}}.png")

https://allmaps.xyz/maps/a4c21a56d19cf41c/{z}/{x}/{y}.png


In [20]:
# use Allmaps API to parse annotations using Map IDs

apiEndpoint = f'https://api.allmaps.org/maps/{mapID}'

# the manifest ID is embedded in the map's own _allmaps metadata, there is no /manifests sub-route on /maps/{id}
mapInfo = requests.get(apiEndpoint).json()
manifestID = mapInfo["_allmaps"]["image"]["canvases"][0]["manifests"][0]["id"].rsplit("/", 1)[-1]

print("Get point, polygon, and metadata for given Map ID: "+apiEndpoint+"\n\r")
print("Get Manifest metadata for given Manifest ID: https://api.allmaps.org/manifests/"+manifestID)


Get point, polygon, and metadata for given Map ID: https://api.allmaps.org/maps/a4c21a56d19cf41c

Get Manifest metadata for given Manifest ID: https://api.allmaps.org/manifests/cd78c03407a95d51
